In [ ]:
import torch
from torch.utils.data import DataLoader
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
import pickle
import neuralgcm
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

from dataloader import NeuralGCMDataset, jax_collate_fn

# ================= 用户配置 =================

MODEL_PATH = '/nfs/gpu_homes/gpu09/home/zhangjing/Code/NeuralGCM/pkl/neuralgcm_04_30_2024_neural_gcm_dynamic_forcing_deterministic_1_4_deg.pkl'
DATA_DIR = '/path/to/your/clean_data_nc'
BATCH_SIZE = 4
PREDICTION_STEPS = 48
NUM_WORKERS = 4

# 区域评估配置
REGION_NAME = "Global"
LAT_RANGE = (-90, 90)
LON_RANGE = (0, 360)

# 评估变量
LEVELS_TO_PLOT = [50, 500, 850, 1000]
EVAL_VARS = ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']

# ================= 辅助函数 =================

def create_region_weight_mask(model_coords, lat_range, lon_range):
    """创建加权掩码 (同前)"""
    lats = model_coords.horizontal.latitudes
    lons = model_coords.horizontal.longitudes
    
    lat_mask = (lats >= lat_range[0]) & (lats <= lat_range[1])
    lon_mask = (lons >= lon_range[0]) & (lons <= lon_range[1])
    region_mask = lat_mask[:, None] & lon_mask[None, :]
    
    weights = np.cos(np.deg2rad(lats))
    weights_2d = weights[:, None] * np.ones_like(lons)[None, :]
    final_weights = weights_2d * region_mask
    
    return final_weights

def compute_weighted_rmse(pred_dict, target_dict, weights, vars_to_eval, prefix=''):
    """
    通用加权 RMSE 计算函数
    prefix: 用于从 pred_dict 中提取变量的前缀 (例如 '_physics_')
    """
    rmse_results = {}
    
    for var in vars_to_eval:
        # 构造带前缀的键名 (如 '_physics_temperature')
        pred_key = f"{prefix}{var}" if prefix else var
        
        if pred_key not in pred_dict or var not in target_dict:
            continue
            
        pred = pred_dict[pred_key] # (B, T, [Z], Lat, Lon)
        target = target_dict[var]  # 真值始终用原始变量名
        
        sq_err = (pred - target)**2
        
        # 计算加权平均
        if pred.ndim == 5: # 3D变量
            w = weights[None, None, None, :, :]
            spatial_sum = (sq_err * w).sum(axis=(-2, -1))
            spatial_mean = spatial_sum / w.sum()
        else: # 2D变量
            w = weights[None, None, :, :]
            spatial_sum = (sq_err * w).sum(axis=(-2, -1))
            spatial_mean = spatial_sum / w.sum()
            
        rmse_results[var] = np.sqrt(spatial_mean) # (B, T, [Z])
        
    return rmse_results

# ================= 主程序 =================

def main():
    # 1. 加载并配置模型
    print(f"Loading model...")
    with open(MODEL_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    model = neuralgcm.PressureLevelModel.from_checkpoint(ckpt)
    
    # === 关键修改: 启用物理核心输出 ===
    print("Enabling Physics Core output...")
    # 这会修改 gin 配置并返回新模型实例
    model = model.with_physics_core_output(enable=True)
    
    # 2. 准备评估 Mask
    region_weights = create_region_weight_mask(model.data_coords, LAT_RANGE, LON_RANGE)
    
    # 3. 编译 JAX 函数
    print("Compiling JAX functions...")
    batched_encode = jax.jit(jax.vmap(model.encode))
    
    inner_steps = 1
    timedelta = np.timedelta64(1, 'h') * inner_steps
    
    def unroll_wrapper(state, forcings):
        return model.unroll(
            state, forcings, steps=PREDICTION_STEPS, timedelta=timedelta, start_with_input=True
        )
    batched_unroll = jax.jit(jax.vmap(unroll_wrapper))
    
    # 4. DataLoader
    dataset = NeuralGCMDataset(DATA_DIR, model, PREDICTION_STEPS)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False, 
        collate_fn=jax_collate_fn, num_workers=NUM_WORKERS, drop_last=False
    )
    
    # 5. 推理循环
    print(f"Start Inference...")
    
    # 存储两份结果
    results_full = []   # NeuralGCM 完整结果
    results_phys = []   # Physics Core 结果
    
    for batch_idx, batch in tqdm(enumerate(loader), total=len(loader)):
        inputs = batch['inputs']
        input_forcings = batch['input_forcings']
        future_forcings = batch['future_forcings']
        targets = batch['targets']
        current_bs = len(batch['init_time_val'])
        
        # Inference
        rng_keys = jax.random.split(jax.random.PRNGKey(batch_idx), current_bs)
        encoded_state = batched_encode(inputs, input_forcings, rng_keys)
        
        # Unroll 会返回包含 '_physics_' 变量的大字典
        _, preds_dict = batched_unroll(encoded_state, future_forcings)
        
        # --- 评估完整模型 (无前缀) ---
        batch_rmse_full = compute_weighted_rmse(
            preds_dict, targets, region_weights, EVAL_VARS, prefix=''
        )
        results_full.append(batch_rmse_full)
        
        # --- 评估物理核心 (前缀 '_physics_') ---
        batch_rmse_phys = compute_weighted_rmse(
            preds_dict, targets, region_weights, EVAL_VARS, prefix='_physics_'
        )
        results_phys.append(batch_rmse_phys)

    # 6. 结果汇总与绘图
    print("Aggregating results...")
    
    def aggregate_results(res_list):
        final = {}
        for var in EVAL_VARS:
            if var not in res_list[0]: continue
            all_rmse = np.concatenate([res[var] for res in res_list], axis=0)
            # (N, T, Z) -> Mean -> (T, Z)
            final[var] = np.mean(all_rmse, axis=0)
        return final

    final_rmse_full = aggregate_results(results_full)
    final_rmse_phys = aggregate_results(results_phys)

    # --- Plotting Comparison ---
    model_levels = model.data_coords.vertical.centers
    level_indices = [np.argmin(np.abs(model_levels - p)) for p in LEVELS_TO_PLOT]
    lead_times = np.arange(PREDICTION_STEPS + 1)
    
    fig, axes = plt.subplots(
        nrows=len(EVAL_VARS), 
        ncols=len(LEVELS_TO_PLOT), 
        figsize=(5 * len(LEVELS_TO_PLOT), 4 * len(EVAL_VARS)),
        constrained_layout=True
    )
    
    for row_idx, var_name in enumerate(EVAL_VARS):
        # 检查是否存在数据
        if var_name not in final_rmse_full: continue
        
        data_full = final_rmse_full[var_name]
        data_phys = final_rmse_phys[var_name]
        is_surface = (data_full.ndim == 1)
        
        for col_idx, (level_val, level_idx) in enumerate(zip(LEVELS_TO_PLOT, level_indices)):
            ax = axes[row_idx, col_idx] if len(EVAL_VARS) > 1 else axes[col_idx]
            
            if is_surface:
                # 地面变量 (只画一次)
                ax.plot(lead_times, data_full, 'r-', label='NeuralGCM', lw=2)
                ax.plot(lead_times, data_phys, 'b--', label='Physics Only', lw=2)
                ax.set_title(f"{var_name} (Surface)")
            else:
                # 高空变量
                d_full = data_full[:, level_idx]
                d_phys = data_phys[:, level_idx]
                
                ax.plot(lead_times, d_full, 'r-', label='NeuralGCM')
                ax.plot(lead_times, d_phys, 'b--', label='Physics Only')
                ax.set_title(f"{var_name} @ {level_val}hPa")
            
            # 计算改进百分比 (在最后时刻)
            if not is_surface:
                imp = (d_phys[-1] - d_full[-1]) / d_phys[-1] * 100
                ax.text(0.05, 0.95, f"Improv: {imp:.1f}%", transform=ax.transAxes, 
                        verticalalignment='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

            ax.set_xlabel("Forecast Hour")
            ax.set_ylabel("RMSE")
            ax.grid(True, alpha=0.3)
            if row_idx == 0 and col_idx == 0:
                ax.legend()
            
    plt.suptitle(f"NeuralGCM vs Physics Core ({REGION_NAME})", fontsize=16)
    plt.savefig('comparison_result.png', dpi=150)
    plt.show()

if __name__ == "__main__":
    main()